###Complete Guide to Fine-Tuning a Sentiment Analysis Model

# What is Fine-Tuning?
1. Fine-tuning is the process of taking a pre-trained model (one that's already been trained on a large dataset) and further training it on a smaller, specific dataset for your particular task.
This allows you to leverage the general knowledge of the pre-trained model while adapting it to perform well on your specific problem.
2. Why Fine-Tune?
There are several compelling reasons to fine-tune a pre-trained model:

  Efficiency: It requires less time, data, and computing resources than training a model from scratch
  
  Better performance: Fine-tuned models often perform better on specific tasks than general models
  
  Domain adaptation: You can adapt a general model to understand the nuances of a specific domain (like financial text)

##Step 1: Set Up Your Environment

Install required packages

Import necessary libraries

In [2]:
!pip install transformers datasets scikit-learn gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      S

In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid

#Step 2: Load the Dataset
1. Dataset Preparation

  The notebook uses the Financial PhraseBank dataset, which contains financial news sentences labeled with sentiment (positive, neutral, negative)

  The data is split into training (70%), validation (15%), and test (15%) sets

  The text is tokenized (converted to numerical representations that the model can understand)



    Load the Dataset Automatically

In [4]:
from datasets import load_dataset

# This line automatically downloads the dataset
dataset = load_dataset("financial_phrasebank", "sentences_allagree")

# Check that it loaded correctly
print(dataset)
print(dataset["train"][0])  # View the first example

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

financial_phrasebank.py:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

The repository for financial_phrasebank contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/financial_phrasebank.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


FinancialPhraseBank-v1.0.zip:   0%|          | 0.00/682k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2264 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 2264
    })
})
{'sentence': 'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .', 'label': 1}


### When you run this code, the dataset will be downloaded automatically from Hugging Face and loaded into your Colab environment.

Potential Issues and Solutions:

"Dataset not found" error

Solution: Make sure you have internet access and that there are no typos in the dataset name.


Slow download

Solution: The first download might take a few minutes. Be patient.


Permission errors

Solution: If you're running Colab on a restricted network, you might need to try from a different network.


"Not enough disk space" error

Solution: Free up space in your Colab instance by removing unnecessary files or restart your Colab runtime.


If the dataset structure is different than expected

Run print(dataset) to see the actual structure and adjust your code accordingly.

##Step 3: Prepare the Dataset

1. Split the dataset

In [5]:
# Split the dataset into train and test sets (80/20 split)
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

2. Adding Validation Set Implementation

In [6]:
# With this three-way split:
train_test_dataset = dataset["train"].train_test_split(test_size=0.3, seed=42)
test_valid_dataset = train_test_dataset["test"].train_test_split(test_size=0.5, seed=42)

# Create a new dictionary with our splits
dataset = {
    "train": train_test_dataset["train"],
    "validation": test_valid_dataset["train"],
    "test": test_valid_dataset["test"]
}

print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print(f"Test size: {len(dataset['test'])}")

Train size: 1267
Validation size: 272
Test size: 272


3. Tokenize the dataset

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True)

#tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = {}
for split in dataset:
    tokenized_datasets[split] = dataset[split].map(tokenize_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

Map:   0%|          | 0/272 [00:00<?, ? examples/s]

Map:   0%|          | 0/272 [00:00<?, ? examples/s]

#2. Model Selection

The notebook uses DistilBERT, a lightweight version of BERT that maintains most of its performance while being faster to train


This is a good choice because it's efficient while still powerful enough for sentiment analysis

##Step 4: Set Up the Model

1. Load the pre-trained model

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2. Define the metrics function


In [9]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

#3. Training Setup

The training arguments include learning rate, batch size, and number of epochs
Evaluation metrics (accuracy and F1 score) are defined to measure performance

##Step 5: Set Up Training Arguments
1. Define training arguments

In [13]:
import transformers
from transformers import EarlyStoppingCallback
print(transformers.__version__)

4.51.3


In [15]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    # Remove these parameters if they're not supported
    # evaluation_strategy="epoch",
    # save_strategy="epoch",
    # load_best_model_at_end=True,
    # metric_for_best_model="f1",
)

##Step 6: Train the Model

1. Create the Trainer

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],  # Use validation set
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    # Remove this if not supported
    # callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

<ipython-input-16-19090fa216d4>:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


2. Start training

In [17]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nilayraut (nilayraut-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss


TrainOutput(global_step=240, training_loss=0.299012279510498, metrics={'train_runtime': 195.8323, 'train_samples_per_second': 19.409, 'train_steps_per_second': 1.226, 'total_flos': 503517561652224.0, 'train_loss': 0.299012279510498, 'epoch': 3.0})

354866218260fb41c9851f32df3ac2c820fef6cf

##Step 7: Evaluate the Model

1. Run evaluation

In [18]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 0.1176069900393486, 'eval_accuracy': 0.9632352941176471, 'eval_f1': 0.9476935899311512, 'eval_runtime': 4.0746, 'eval_samples_per_second': 66.754, 'eval_steps_per_second': 4.172, 'epoch': 3.0}


2. Error analysis

In [19]:
# Get predictions for test samples
test_samples = tokenized_datasets["test"].select(range(10))  # Use select to get the first 10 samples
raw_pred, labels, _ = trainer.predict(test_samples)
predictions = raw_pred.argmax(-1)

# Compare predictions with actual labels
for i, (pred, actual) in enumerate(zip(predictions, labels)):
    text = dataset["test"][i]["sentence"]

    print(f"Text: {text}")
    print(f"Predicted: {pred} | Actual: {actual}")
    print("-" * 50)

Text: Taking a cue from the playbook of the East Dillon Lions , we 've created a special team of heavy-hitting style players , such as boot-cut jeans , tummy tops and , of course , cowboy boots .
Predicted: 1 | Actual: 1
--------------------------------------------------
Text: Kone shares dropped 4.1 percent to  x20ac 43 US$ 55.77 in Helsinki .
Predicted: 0 | Actual: 0
--------------------------------------------------
Text: National sponsors for The Big Read include National Endowment for the Arts in cooperation with the Institute of Museum and Library Services and Arts Midwest .
Predicted: 1 | Actual: 1
--------------------------------------------------
Text: The issue came up in connection with discussion with local municipalities concerning the sale of water to industrial facilities .
Predicted: 1 | Actual: 1
--------------------------------------------------
Text: The world 's second largest stainless steel maker said net profit in the three-month period until Dec. 31 surged to eu

#4. Hyperparameter Optimization

Multiple combinations of learning rates, batch sizes, and weight decay values are tested

This helps find the optimal configuration for the model



##Step 8: Hyperparameter Optimization

In [20]:
print("Starting hyperparameter optimization...")

# Define a simplified hyperparameter grid
param_grid = {
    'learning_rate': [1e-5, 3e-5],
    'batch_size': [8, 16],
    'weight_decay': [0.01, 0.1],
}

# Initialize results tracking
results = []

# Grid search - simplified version for older transformers
for lr in param_grid['learning_rate']:
    for bs in param_grid['batch_size']:
        for wd in param_grid['weight_decay']:
            print(f"\nTraining with lr={lr}, batch_size={bs}, weight_decay={wd}")

            # Set up training arguments (compatible with older versions)
            training_args = TrainingArguments(
                output_dir=f"./results_lr{lr}_bs{bs}_wd{wd}",
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=bs,
                num_train_epochs=3,
                weight_decay=wd,
            )

            # Reset model for each run
            model = AutoModelForSequenceClassification.from_pretrained(
                "distilbert-base-uncased",
                num_labels=3
            )

            # Create trainer
            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=tokenized_datasets["train"],
                eval_dataset=tokenized_datasets["validation"],
                tokenizer=tokenizer,
                compute_metrics=compute_metrics
            )

            # Train the model
            trainer.train()

            # Evaluate
            eval_result = trainer.evaluate()

            # Store results
            results.append({
                'learning_rate': lr,
                'batch_size': bs,
                'weight_decay': wd,
                'accuracy': eval_result['eval_accuracy'],
                'f1': eval_result['eval_f1']
            })

            print(f"Results: Accuracy = {eval_result['eval_accuracy']:.4f}, F1 = {eval_result['eval_f1']:.4f}")

# Print results in a tabular format
print("\nAll hyperparameter results:")
for result in results:
    print(f"lr={result['learning_rate']}, bs={result['batch_size']}, wd={result['weight_decay']}: Accuracy={result['accuracy']:.4f}, F1={result['f1']:.4f}")

# Find best configuration
best_result = max(results, key=lambda x: x['f1'])
print(f"\nBest hyperparameters:")
print(f"Learning rate: {best_result['learning_rate']}")
print(f"Batch size: {best_result['batch_size']}")
print(f"Weight decay: {best_result['weight_decay']}")
print(f"Best F1 Score: {best_result['f1']:.4f}")
print(f"Best Accuracy: {best_result['accuracy']:.4f}")

# Train final model with best hyperparameters
print("\nTraining final model with best hyperparameters...")
final_training_args = TrainingArguments(
    output_dir="./final_model",
    learning_rate=best_result['learning_rate'],
    per_device_train_batch_size=int(best_result['batch_size']),
    per_device_eval_batch_size=int(best_result['batch_size']),
    num_train_epochs=3,
    weight_decay=best_result['weight_decay'],
)

final_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],  # Now evaluate on test set
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

final_trainer.train()
final_results = final_trainer.evaluate()
print(f"Final model performance on test set: Accuracy = {final_results['eval_accuracy']:.4f}, F1 = {final_results['eval_f1']:.4f}")

Starting hyperparameter optimization...

Training with lr=1e-05, batch_size=8, weight_decay=0.01


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-20-9d22bb1e46ee>:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Results: Accuracy = 0.9559, F1 = 0.9361

Training with lr=1e-05, batch_size=8, weight_decay=0.1


...

Step,Training Loss


Results: Accuracy = 0.9706, F1 = 0.9646

All hyperparameter results:
lr=1e-05, bs=8, wd=0.01: Accuracy=0.9559, F1=0.9361
lr=1e-05, bs=8, wd=0.1: Accuracy=0.9596, F1=0.9401
lr=1e-05, bs=16, wd=0.01: Accuracy=0.9191, F1=0.8859
lr=1e-05, bs=16, wd=0.1: Accuracy=0.9191, F1=0.8859
lr=3e-05, bs=8, wd=0.01: Accuracy=0.9816, F1=0.9758
lr=3e-05, bs=8, wd=0.1: Accuracy=0.9816, F1=0.9758
lr=3e-05, bs=16, wd=0.01: Accuracy=0.9706, F1=0.9646
lr=3e-05, bs=16, wd=0.1: Accuracy=0.9706, F1=0.9646

Best hyperparameters:
Learning rate: 3e-05
Batch size: 8
Weight decay: 0.01
Best F1 Score: 0.9758
Best Accuracy: 0.9816

Training final model with best hyperparameters...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-20-9d22bb1e46ee>:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  final_trainer = Trainer(


Step,Training Loss


Final model performance on test set: Accuracy = 0.9743, F1 = 0.9628


Results showed that a learning rate of 3e-5, batch size of 8, and weight decay of 0.01 performed best

##Step 9: Save the Model

1. Save the model and tokenizer

In [21]:
model_path = "./final_sentiment_model"
final_trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

('./final_sentiment_model/tokenizer_config.json',
 './final_sentiment_model/special_tokens_map.json',
 './final_sentiment_model/vocab.txt',
 './final_sentiment_model/added_tokens.json',
 './final_sentiment_model/tokenizer.json')

##Step 10: Create Inference Pipeline

1. Load the saved model for inference

In [22]:
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path)

2. Create prediction function

In [23]:
def predict_sentiment(text):
    inputs = loaded_tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = loaded_model(**inputs)

    scores = torch.softmax(outputs.logits, dim=1)
    prediction = scores.argmax(1).item()

    sentiment_labels = ["negative", "neutral", "positive"]
    return sentiment_labels[prediction]

3. Test the prediction function

In [24]:
test_text = "Company profits exceeded expectations."
print(f"Text: {test_text}")
print(f"Sentiment: {predict_sentiment(test_text)}")

Text: Company profits exceeded expectations.
Sentiment: positive


#5. Model Evaluation

The fine-tuned model is evaluated on a held-out test set

It's compared to the baseline (pre-fine-tuned) model

##Step 11: Compare with Baseline Model

In [26]:
print("Comparing fine-tuned model with baseline (pre-fine-tuned) model...")

# First, let's check the actual structure of your dataset
print("Dataset type:", type(dataset["test"]))
print("First item in test dataset:", dataset["test"][0])  # See what the structure looks like

# Define evaluation function with fixed data access
def evaluate_model(model, dataset_split, tokenizer, device="cpu"):
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    # Access dataset items properly based on structure
    for i in range(len(dataset_split)):
        # Get the text and label based on your dataset structure
        if hasattr(dataset_split, "features") and "sentence" in dataset_split.features:
            # Dataset is a Dataset object with features
            text = dataset_split[i]["sentence"]
            label = dataset_split[i]["label"]
        elif isinstance(dataset_split[i], dict) and "sentence" in dataset_split[i]:
            # Dataset item is a dictionary with 'sentence' key
            text = dataset_split[i]["sentence"]
            label = dataset_split[i]["label"]
        else:
            # Try this structure as a fallback
            text = dataset_split[i][0]  # Assuming text is the first item
            label = dataset_split[i][1]  # Assuming label is the second item

        # Tokenize single example
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()

        all_preds.append(pred)
        all_labels.append(label)

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return {
        "accuracy": accuracy,
        "f1": f1
    }

# Let's try a different approach to access the dataset
try:
    # Load a fresh pre-trained model (without fine-tuning)
    baseline_model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=3
    )

    # Sample a small subset for testing to make sure our function works
    test_subset = dataset["test"].select(range(10)) if hasattr(dataset["test"], "select") else dataset["test"][:10]
    print("Testing evaluation function on small subset first...")
    test_results = evaluate_model(baseline_model, test_subset, tokenizer)
    print(f"Test subset results: Accuracy = {test_results['accuracy']:.4f}, F1 = {test_results['f1']:.4f}")

    # Now evaluate on the full test set
    print("Evaluating baseline model (pre-fine-tuning)...")
    baseline_results = evaluate_model(baseline_model, dataset["test"], tokenizer)

    # Evaluate fine-tuned model
    print("Evaluating fine-tuned model...")
    finetuned_results = evaluate_model(loaded_model, dataset["test"], loaded_tokenizer)

    # Print comparison
    print("\nModel Performance Comparison:")
    print("-" * 50)
    print(f"Metric    | Baseline (Pre-Fine-Tuned) | Fine-Tuned")
    print(f"Accuracy  | {baseline_results['accuracy']:.4f}                | {finetuned_results['accuracy']:.4f}")
    print(f"F1 Score  | {baseline_results['f1']:.4f}                | {finetuned_results['f1']:.4f}")
    print(f"Improvement in Accuracy: {(finetuned_results['accuracy'] - baseline_results['accuracy'])*100:.2f}%")
    print(f"Improvement in F1 Score: {(finetuned_results['f1'] - baseline_results['f1'])*100:.2f}%")

except Exception as e:
    print(f"Error during evaluation: {e}")
    print("\nLet's try an alternative approach:")

    # Alternative approach using a simpler evaluation method
    def simple_evaluate(model, dataset_split, tokenizer, device="cpu"):
        model.to(device)
        model.eval()

        correct = 0
        total = 0

        # Create a very simple access function that tries multiple approaches
        def get_item(item_index):
            try:
                # First try direct access
                return dataset_split[item_index]["sentence"], dataset_split[item_index]["label"]
            except:
                try:
                    # Try accessing as a Dataset feature
                    return dataset_split["sentence"][item_index], dataset_split["label"][item_index]
                except:
                    # Try as a tuple
                    return dataset_split[item_index][0], dataset_split[item_index][1]

        # Process one by one
        for i in range(len(dataset_split)):
            try:
                text, label = get_item(i)

                inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
                inputs = {k: v.to(device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model(**inputs)
                    pred = torch.argmax(outputs.logits, dim=1).item()

                if pred == label:
                    correct += 1
                total += 1
            except Exception as e:
                print(f"Error processing item {i}: {e}")
                continue

        accuracy = correct / total if total > 0 else 0
        return {"accuracy": accuracy}

    # Try the simpler evaluation
    print("Using simplified evaluation...")
    base_acc = simple_evaluate(baseline_model, dataset["test"], tokenizer)
    ft_acc = simple_evaluate(loaded_model, dataset["test"], loaded_tokenizer)

    print("\nSimplified Model Performance Comparison:")
    print("-" * 50)
    print(f"Metric    | Baseline (Pre-Fine-Tuned) | Fine-Tuned")
    print(f"Accuracy  | {base_acc['accuracy']:.4f}                | {ft_acc['accuracy']:.4f}")
    print(f"Improvement in Accuracy: {(ft_acc['accuracy'] - base_acc['accuracy'])*100:.2f}%")

Comparing fine-tuned model with baseline (pre-fine-tuned) model...
Dataset type: <class 'datasets.arrow_dataset.Dataset'>
First item in test dataset: {'sentence': "Taking a cue from the playbook of the East Dillon Lions , we 've created a special team of heavy-hitting style players , such as boot-cut jeans , tummy tops and , of course , cowboy boots .", 'label': 1}


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Testing evaluation function on small subset first...
Test subset results: Accuracy = 0.1000, F1 = 0.0606
Evaluating baseline model (pre-fine-tuning)...
Evaluating fine-tuned model...

Model Performance Comparison:
--------------------------------------------------
Metric    | Baseline (Pre-Fine-Tuned) | Fine-Tuned
Accuracy  | 0.1140                | 0.9743
F1 Score  | 0.0682                | 0.9628
Improvement in Accuracy: 86.03%
Improvement in F1 Score: 89.45%


#**Results show a dramatic improvement: from 11.4% to 97.4% accuracy, an 86% absolute improvement**

#6. Inference Pipeline

A function is created to use the model for predictions on new text

A Gradio interface is built to provide a user-friendly way to test the model

##Step 12: Create a Demo with Gradio

1. Set up a Gradio interface

In [34]:
def gradio_predict(text):
    return predict_sentiment(text)

# Define examples to display in the interface, explicitly organized by sentiment
examples = [
    # Positive examples
    ["Our company exceeded quarterly revenue targets by 18%, leading to our highest profit margin in five years."],
    ["The board has approved a 15% increase in dividend payments to shareholders effective next quarter."],
    ["Our innovative cost reduction program has improved operating margins by 340 basis points year-over-year."],
    ["The acquisition has exceeded synergy targets, delivering $45M in annual savings versus the projected $30M."],
    ["Our new digital banking platform has attracted over 500,000 customers in its first quarter, well above expectations."],

    # Neutral examples
    ["The company has maintained its previous guidance for the upcoming fiscal year."],
    ["Our Q3 earnings report shows results largely aligned with analyst consensus estimates."],
    ["The CFO announced that the company will refinance its existing debt at current market rates."],
    ["The regulatory review of our pending merger is proceeding as scheduled with no reported concerns."],
    ["The board has appointed Sarah Johnson as the new Chief Technology Officer, effective next month."],

    # Negative examples
    ["Rising raw material costs have reduced our gross margins by 380 basis points compared to last year."],
    ["Due to ongoing supply chain disruptions, we expect to miss our delivery targets for the next two quarters."],
    ["The company will close five underperforming retail locations as part of our cost-cutting initiative."],
    ["Our market share has declined by 3.5 percentage points following the entry of three new competitors."],
    ["The failed product launch has resulted in a $75 million inventory write-down for this quarter."]
]

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Textbox(lines=5, placeholder="Enter financial text here..."),
    outputs="text",
    title="Financial Sentiment Analysis",
    description="Enter a financial statement to predict its sentiment (positive, neutral, or negative).",
    examples=examples,
    theme="default"
)

demo.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://555cd73d6fd4d7f284.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#Results and Insights
1. The notebook demonstrates how effective fine-tuning can be. The pre-trained DistilBERT model started with only about 11% accuracy on the financial sentiment task, but after fine-tuning, it achieved over 97% accuracy.

2. This dramatic improvement shows that while pre-trained models have general language understanding, they need domain-specific training to excel at particular tasks like financial sentiment analysis.
3. The Gradio interface provides a practical way to interact with the model, allowing users to input financial statements and receive sentiment predictions (positive, neutral, or negative).

This approach can be used for many different text classification tasks beyond sentiment analysis, such as:

1. Content categorization

2. Intent detection

3. Spam detection

4. Brand monitoring

5. Customer feedback analysis